In [102]:
import torch

from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, IterableDataset

import json
import random
from pyarrow import parquet as pq

In [103]:
class LoadLanguageData(IterableDataset):
    
    def __init__(self, path:str, columns:list) -> None:

        super().__init__()

        self.cols = columns
        self.path = path
        self.dataset = pq.ParquetDataset(self.path)

    def __iter__(self):

        for fragment in self.dataset.fragments:

            table = fragment.to_table()
            rows = table.to_pylist()

            for row in rows:

                yield {
                    'input_ids' : torch.tensor(row[self.cols[1]], dtype=torch.long),
                    'output_ids': torch.tensor(row[self.cols[2]], dtype=torch.long),
                    'target_ids': torch.tensor(row[self.cols[3]], dtype=torch.long)
                }

def wrapper_collate_batch(pad_value_in, pad_value_out):

    def collate_batch(batch):

        input_list  = [item['input_ids']  for item in batch]
        output_list = [item['output_ids'] for item in batch]
        target_list = [item['target_ids'] for item in batch]

        input_padded  = pad_sequence(input_list,  batch_first=True, padding_value=pad_value_in)
        output_padded = pad_sequence(output_list, batch_first=True, padding_value=pad_value_out)
        target_padded = pad_sequence(target_list, batch_first=True, padding_value=pad_value_out)

        return (input_padded, output_padded, target_padded)

    return collate_batch


def load_tokenizer(path:str) -> tuple:

    with open(path) as f:
        tokenizer = json.load(f)
    
    index2word = {int(key): value for key, value in tokenizer["inverse_vocab"].items()}
    special_tokens = tokenizer["special_tokens"]

    return special_tokens, index2word


def index2word(input_tensor: torch.Tensor, vocab_mapper: dict, unk_token:int) -> list:

    indices_matrix = input_tensor.cpu().numpy()

    output = [
        [vocab_mapper.get(idx, vocab_mapper[unk_token]) for idx in sentence]
        for sentence in indices_matrix
    ]
    
    return output

In [104]:
sp_tokens_eng, vocab_idx2word_eng = load_tokenizer("../data/main/vocabs/eng_tokenizer.json")
sp_tokens_spa, vocab_idx2word_spa = load_tokenizer("../data/main/vocabs/spa_tokenizer.json")

pad_value_eng = sp_tokens_eng["[PAD]"]
pad_value_spa = sp_tokens_spa["[PAD]"]

In [105]:
train_path = "../data/main/spa-eng/train_spa_eng"
train_schema = pq.ParquetDataset(train_path)
train_cols = train_schema.schema.names

test_path = "../data/main/spa-eng/test_spa_eng"
test_schema = pq.ParquetDataset(test_path)
test_cols = test_schema.schema.names

In [106]:
train_data = LoadLanguageData(train_path, train_cols)
train_loader = DataLoader(train_data, batch_size=64, collate_fn=wrapper_collate_batch(pad_value_eng, pad_value_spa))

test_data = LoadLanguageData(test_path, test_cols)
test_loader = DataLoader(test_data, batch_size=64, collate_fn=wrapper_collate_batch(pad_value_eng, pad_value_spa))

In [107]:
import numpy as np

for eng_in, spa_out, spa_t in train_loader:
    break

print(np.array(index2word(eng_in, vocab_idx2word_eng, sp_tokens_eng["[UNK]"]))[:3], end="\n\n\n")
print(np.array(index2word(spa_out,vocab_idx2word_spa, sp_tokens_spa["[UNK]"]))[:3], end="\n\n\n")
print(np.array(index2word(spa_t,  vocab_idx2word_spa, sp_tokens_spa["[UNK]"]))[:3])

[['go' 'make' 'popcorn' '.' '[END]' '[PAD]' '[PAD]' '[PAD]' '[PAD]'
  '[PAD]' '[PAD]']
 ['there' 'were' 'no' 'radios' 'in' 'those' 'days' '.' '[END]' '[PAD]'
  '[PAD]']
 ['there' "'s" 'a' 'small' 'pond' 'in' 'our' 'garden' '.' '[END]' '[PAD]']]


[['[START]' 'andá' 'a' 'hacer' 'pochoclo' '.' '[PAD]' '[PAD]' '[PAD]'
  '[PAD]' '[PAD]' '[PAD]']
 ['[START]' 'no' 'había' 'radios' 'aún' 'entonces' '.' '[PAD]' '[PAD]'
  '[PAD]' '[PAD]' '[PAD]']
 ['[START]' 'hay' 'un' 'pequeño' 'estanque' 'en' 'nuestro' 'jardín' '.'
  '[PAD]' '[PAD]' '[PAD]']]


[['andá' 'a' 'hacer' 'pochoclo' '.' '[END]' '[PAD]' '[PAD]' '[PAD]'
  '[PAD]' '[PAD]' '[PAD]']
 ['no' 'había' 'radios' 'aún' 'entonces' '.' '[END]' '[PAD]' '[PAD]'
  '[PAD]' '[PAD]' '[PAD]']
 ['hay' 'un' 'pequeño' 'estanque' 'en' 'nuestro' 'jardín' '.' '[END]'
  '[PAD]' '[PAD]' '[PAD]']]


In [108]:
class Encoder(nn.Module):
    def __init__(self, vocab_size:int, embedding_size:int, hidden_size:int, layers:int, p_dropout:float) -> None:
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_size)
        self.dropout = nn.Dropout(p_dropout)

        self.gru = nn.GRU(embedding_size, hidden_size, layers, bidirectional=True, batch_first=True)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:

        x = self.embedding(x)
        x = self.dropout(x)

        # output will have shape
        # (batch, seq_length, hidden_size * 2 if bidireccional else hidden_size)
        # hidden will have shape 
        # (layer * 2 if bidireccional else layer, batch_size, hidden_size)
        outputs, hidden = self.gru(x)

        # (batch, 1, hidden_size + hidden_size)
        x = torch.concat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)

        # Expected hidden dimension (e.g if it'll be used for another GRU)
        x = torch.unsqueeze(x, dim=0)

        # output for cross-attention
        # x as hidden state
        return outputs, x


class DotProductAttention(nn.Module):
    def __init__(self) -> None:
        super().__init__()

    # Cross-Attention
    def forward(self, hidden:torch.Tensor, encoder_outputs:torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:

        # Q -> hidde_state del Decoder (primera iteración hidden del encoder)
        # K y V -> output del Encoder
        Q = hidden.transpose(0, 1) # (batch, 1, encoder_outputs.size(-1))
        K = encoder_outputs
        K_t = K.transpose(1, 2) # (batch, encoder_outputs.size(-1), seq_len)

        d_k = Q.size(-1)
        energy_scaled = torch.bmm(Q, K_t) / (d_k ** 0.5)

        att_weights = torch.softmax(energy_scaled, dim=2)
        context = torch.bmm(att_weights, K).squeeze(1)

        return att_weights, context


class BridgeEncoderDecoder(nn.Module):
    def __init__(self, encoder_hidden_size, decoder_hidden_size):
        super().__init__()

        # Proyection to match hidden_state and output between encoder-decoder
        self.bridge_hidden  = nn.Linear(encoder_hidden_size, decoder_hidden_size)
        self.output_proyect = nn.Linear(encoder_hidden_size, decoder_hidden_size)
    
    def forward(self, encoder_hidden_state, encoder_outputs):

        hidden_proyection = torch.tanh(self.bridge_hidden(encoder_hidden_state))
        output_proyection = self.output_proyect(encoder_outputs)

        return hidden_proyection, output_proyection


class Decoder(nn.Module):
    def __init__(self, vocab_size:int, embedding_size:int, hidden_size:int, attention:DotProductAttention, p_dropout:float) -> None:
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_size)
        self.dropout = nn.Dropout(p_dropout)

        self.attention = attention
    
        self.gru = nn.GRU(embedding_size + hidden_size, hidden_size, batch_first=True)
        # GRU [output] will be concatenated with [context] and [embedding] for more information
        # It's a Skip Connection like Deep Output Layer or Output Fusion.
        self.ol = nn.Linear(hidden_size + hidden_size + embedding_size, vocab_size)

    def forward(self, x:torch.Tensor, hidden_state:torch.Tensor, encoder_outputs:torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:

        x = x.unsqueeze(1) # (batch, 1)

        embedded = self.embedding(x)
        embedded = self.dropout(embedded) # (batch, 1, embedding_size)

        _, context = self.attention(hidden_state, encoder_outputs)
        context = context.unsqueeze(1) # (batch, 1, hidden_size)

        gru_input = torch.concatenate((embedded, context), dim=2) # (batch, 1, embedding_size + hidden_size)
        output, hidden = self.gru(gru_input, hidden_state)

        prediction = torch.concat((output, context, embedded), dim=2)
        prediction = self.ol(prediction).squeeze(1)

        return output, hidden, prediction


class Seq2Seq:
    def __init__(self, encoder, bridge, decoder) -> None:
        super().__init__()

        self.encoder = encoder
        self.bridge = bridge
        self.decoder = decoder

    def fit(self, x_1, x_2):

        enc_output, enc_hidden = self.encoder(x_1)
        enc_output_pro, enc_hidden_pro = self.bridge(enc_output, enc_hidden)

        hidden = enc_hidden_pro
        n_tokens = x_2.size(-1)
        for idx_token in range(n_tokens):

            current_batch_tokens = x_2[:, idx_token]
            _, hidden, pred = self.decoder(current_batch_tokens, hidden, enc_output_pro) # In the first iteration we use encoder hidden_state

            # Agregar aquí teacher forcing

In [109]:
# Getting Batch
eng_batch, spa_o_batch, spa_t_batch = next(iter(train_loader))

# Testing Encoder class and dimensions
encoder = Encoder(
    vocab_size=len(vocab_idx2word_eng), 
    embedding_size=300,
    hidden_size=50,
    layers=1,
    p_dropout=0.3
)
outputs_e, hidden_e = encoder(eng_batch)

print(f"Batch: {eng_batch.shape}")
print(f"Output_enc: {outputs_e.shape}")
print(f"Hidden_enc: {hidden_e.shape}")

# Testing Attention with encoder results and dimensions
dot_prod_att = DotProductAttention()
att, con = dot_prod_att(hidden_e, outputs_e)

print("")
print(f"Att Weights: {att.shape}")
print(f"Context: {con.shape}")

# Testing proyection from encoder results to decoder format and dimensions
bridge_conexion = BridgeEncoderDecoder(hidden_e.size(-1), 200)
hidden_pro, outputs_pro = bridge_conexion(hidden_e, outputs_e)

print(hidden_pro.shape)
print(outputs_pro.shape)

Batch: torch.Size([64, 11])
Output_enc: torch.Size([64, 11, 100])
Hidden_enc: torch.Size([1, 64, 100])

Att Weights: torch.Size([64, 1, 11])
Context: torch.Size([64, 100])
torch.Size([1, 64, 200])
torch.Size([64, 11, 200])


In [ ]:
decoder = Decoder(
    vocab_size = len(vocab_idx2word_spa), 
    embedding_size = 300,
    hidden_size = 200,
    attention = DotProductAttention(),
    p_dropout = 0.3
)

n_tokens = spa_o_batch.size(-1)
teacher_forcing_ratio = 0.5
for idx_token in range(n_tokens):

    current_batch_tokens = spa_o_batch[:, idx_token]
    output_dec, hidden_dec, pred = decoder(current_batch_tokens, hidden_pro, outputs_pro) # In the first iteration we use encoder hidden_state

    teacher_force = random.random() < teacher_forcing_ratio
    top = pred.argmax(1)

    input_step = spa_o_batch[:, idx_token + 1] if teacher_force else top

    break

print(output_dec.shape)
print(hidden_dec.shape)
print(pred.shape)

torch.Size([64, 1, 200])
torch.Size([1, 64, 200])
torch.Size([64, 28587])
